# DiffSinger Miadu Fine-tuning (多帳號協作版)
**適用情境**：主帳號擁有大容量儲存空間 (存放 5TB/訓練數據)，小帳號使用免費 T4 算力。

### ⚠️ 執行前必看！(捷徑掛載法)
為了能讓小帳號讀取主帳號的資料，請**必須**先執行以下步驟：
1. 使用小帳號登入 Google Drive，並點開主帳號分享的連結：`https://drive.google.com/drive/folders/1QsB2TIvCfMtSPhrHS9T29YLqR-Fbwhhz?usp=sharing`
2. 在頂部的資料夾名稱旁邊，點擊下拉選單，選擇 **「新增捷徑到雲端硬碟 (Add shortcut to Drive)」**。
3. 將捷徑放在 **「我的雲端硬碟」** 的根目錄，並確保該資料夾捷徑的名稱為 `DiffSinger_Upload_This` (如果名稱不同請重新命名捷徑)。
4. 捷徑建立後，小帳號的 Drive 雖然不會被扣儲存空間，但在 Colab 裡卻能把捷徑當成真實資料夾來讀取！
5. **重要：** 訓練產出的模型檔案 (Checkpoints) 將會自動存回小帳號的雲端硬碟中，不影響主帳號。

## 第一步：掛載小帳號的 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取/更新代碼與環境配置

In [ ]:
%cd /content/
import os
if not os.path.exists('diffsinger-repo'):
    !git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
else:
    %cd diffsinger-repo
    !git pull
    %cd ..

%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0 jieba g2pM pretty_midi textgrid

## 第三步：跨帳號數據同步與路徑對齊

In [ ]:
import os
import shutil

# 這是主帳號的捷徑路徑
drive_root = "/content/drive/MyDrive/DiffSinger_Upload_This"

if not os.path.exists(drive_root):
    raise FileNotFoundError("找不到捷徑！請確認您已在小帳號的 Google Drive 根目錄建立名為 'DiffSinger_Upload_This' 的捷徑！")

# 這是小帳號自己的儲存空間
drive_target = "/content/drive/MyDrive/checkpoints_finetune"
target_link = "/content/diffsinger-repo/checkpoints"

os.makedirs(drive_target, exist_ok=True)
os.makedirs("/content/diffsinger-repo", exist_ok=True)

if os.path.exists(target_link) and not os.path.islink(target_link):
    print("| 正在清理舊的本地 checkpoints 資料夾...")
    shutil.rmtree(target_link)

if not os.path.exists(target_link):
    print("| 建立 Drive Checkpoint 軟連結 (指向小帳號)...")
    os.symlink(drive_target, target_link)

def safe_copy(src, dst):
    if os.path.isdir(src):
        if not os.path.exists(dst):
            os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src):
            safe_copy(os.path.join(src, item), os.path.join(dst, item))
    else:
        if not os.path.exists(os.path.dirname(dst)):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
            print(f"  -> 複製: {os.path.basename(src)}")
            shutil.copy2(src, dst)

def robust_sync(src_dir, dest_path, min_size_mb=1, label="數據"):
    if not os.path.exists(dest_path) or os.path.getsize(dest_path) < min_size_mb * 1024 * 1024:
        print(f"| >>> 正在從捷徑同步 {label} (這可能需要幾分鐘，請稍候)...")
        try:
            safe_copy(src_dir, dest_path)
            print(f"| <<< {label} 同步完成。")
        except OSError as e:
            print(f"\n❌ 同步失敗: {e}")
            print("這通常是因為 Colab 與 Google Drive 連線中斷 (Transport endpoint is not connected)。")
            print("請到上方選單選擇「執行階段」->「重新啟動執行階段」，然後重新執行所有儲存格！")
    else:
        print(f"| ✅ {label} 已存在，跳過同步。")

print('| --- 開始環境同步 ---')
robust_sync(f"{drive_root}/checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt", 
            "/content/diffsinger-repo/checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt", 800, "1117 底模")

robust_sync(f"{drive_root}/checkpoints/nsf_hifigan_44.1k_hop512_128bin_2024.02/model_ckpt_steps_161274.ckpt", 
            "/content/diffsinger-repo/checkpoints/nsf_hifigan_44.1k_hop512_128bin_2024.02/model_ckpt_steps_161274.ckpt", 10, "128bin 聲碼器")

robust_sync(f"{drive_root}/data/binary/miadu_finetune", 
            "/content/diffsinger-repo/data/binary/miadu_finetune", 1000, "二進制訓練數據")

# 如果主帳號有舊的 finetune 紀錄 (如 800 步)，只複製必要的權重檔，忽略幾千個備份的代碼檔！
src_ckpt_dir = f"{drive_root}/checkpoints_finetune/miadu_finetune_v1"
dst_ckpt_dir = f"{drive_target}/miadu_finetune_v1"
if os.path.exists(src_ckpt_dir) and not os.path.exists(os.path.join(dst_ckpt_dir, "config.yaml")):
    print("| >>> 偵測到主帳號有先前的微調紀錄，正在過濾並複製 (僅拷貝權重與設定檔)...")
    os.makedirs(dst_ckpt_dir, exist_ok=True)
    for item in os.listdir(src_ckpt_dir):
        if item.endswith(".ckpt") or item.endswith(".yaml") or "events" in item:
            src_file = os.path.join(src_ckpt_dir, item)
            dst_file = os.path.join(dst_ckpt_dir, item)
            if os.path.isfile(src_file):
                print(f"  -> 複製核心存檔: {item}")
                shutil.copy2(src_file, dst_file)
    print("| <<< 接續訓練環境準備完成。")

print('| --- 數據環境已就緒 ---')
!du -sh /content/diffsinger-repo/data/binary/miadu_finetune

## 第四步：啟動訓練 (微調)

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1

## 第五步：完成與提示

In [ ]:
print("訓練已啟動。Checkpoint 將自動保存在您【小帳號】的 Drive 中。")